In [1]:
import math
import random
import numpy as np
import time
from waveNetArchitecture import Value, Linear, BatchNorm1D, LayerNorm, Tanh, ReLU, Embedding, FlattenConsecutive, Sequential, cross_entropy, BagOfWords, PositionalEmbedding, Head, MultiHead, FeedForward, Block, AdamW
from bpeTokenizer import RegexTokenizer
with open('fineweb_edu_subset.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [2]:
tok = RegexTokenizer.load('shakespearTokenizer.json')
EOS_ID = tok.add_special_token('<|endoftext|>')  # reserved id, never produced by encoding ordinary text
tok.save('shakespearTokenizer.json')  # persist it so the id stays stable across reloads
vocab_size = tok.vocab_size
encode = tok.encode
decode = tok.decode

blockSize = 64 # Increase the amount of context
inputs, outputs = [],[]
print(vocab_size, 'EOS_ID:', EOS_ID)

1025 EOS_ID: 1024


In [3]:
# split back into documents (the fetch script joined them with a blank line) and
# insert EOS_ID after each one, so the model sees an explicit "this document ended"
# signal instead of documents just running into each other
docs = [d for d in text.split('\n\n') if d]
ids = []
for d in docs:
    ids.extend(encode(d))
    ids.append(EOS_ID)
data = np.array(ids) # plain int array: this is data, not a parameter, so it needs no autograd

n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]
print(data[:20], '| train:', len(train_data), 'val:', len(val_data), '| docs:', len(docs))

[354 292 266 509 468 338 972 300 101  10 480 395 267 519  44  32 423 645
 297 855] | train: 3309437 val: 367716 | docs: 1634


In [4]:
n_embd = 32     # embedding dimensionality (must divide evenly across num_heads)
num_heads = 4   # attention heads per block
n_blocks = 4    # how many transformer blocks to stack

model = Sequential([
  Embedding(vocab_size, n_embd),
  PositionalEmbedding(blockSize, n_embd),
  *[Block(n_embd, num_heads, blockSize) for _ in range(n_blocks)],
  LayerNorm(n_embd),
  Linear(n_embd, vocab_size),
])

# keep initial predictions close to uniform, so the starting loss is near -log(1/vocab_size)
model.layers[-1].weight.data *= 0.1

parameters = model.parameters()
print(sum(p.data.size for p in parameters)) # number of parameters in total

119169


In [8]:
max_steps = 30000   # 200k is unreachable on numpy/CPU; raise it once you see the curve still falling
batch_size = 32
eval_every = 1000   # how often to measure held-out loss
eval_iters = 10     # batches averaged per measurement (more = less noisy, but costs forward passes)

lossi = []      # raw per-step training loss. NOT log10 - the plot cell takes the log if it wants one
val_lossi = []  # (step, train_estimate, val_estimate), recorded every eval_every steps
ud = []

opt = AdamW(parameters, lr=1e-3, betas=(0.9,0.95), weight_decay=0.01)

# build sliding-window dataset: each input is `blockSize` consecutive tokens,
# targets are that same window shifted one to the right - a prediction at every position,
# not just one at the end (that's the whole point of keeping T alive through attention)
def make_windows(tokens, blockSize):
    # sliding_window_view returns a *view*, so this is free. The old list-of-arrays
    # version materialized every window: ~560MB for Shakespeare.
    n = len(tokens) - blockSize
    X = np.lib.stride_tricks.sliding_window_view(tokens, blockSize)[:n]      # (n, blockSize) context
    Y = np.lib.stride_tricks.sliding_window_view(tokens[1:], blockSize)[:n]  # (n, blockSize) same, shifted by one
    return X, Y

# built from train_data and val_data separately, so no val window is ever trained on
Xtr, Ytr = make_windows(train_data, blockSize)
Xva, Yva = make_windows(val_data, blockSize)
print(f'train windows: {Xtr.shape[0]:,}   val windows: {Xva.shape[0]:,}')

def split_loss(X, Y, iters=eval_iters):
    """Mean loss over `iters` random batches. Forward pass only - we never call
    .backward() here, so each graph is discarded instead of accumulating grads."""
    losses = []
    for _ in range(iters):
        ix = np.random.randint(0, X.shape[0], (batch_size,))
        losses.append(float(cross_entropy(model(X[ix]), Y[ix]).data))
    return np.mean(losses)

for i in range(max_steps):

    # minibatch construct - drawn from the TRAIN windows only
    ix = np.random.randint(0, Xtr.shape[0], (batch_size,))
    Xb, Yb = Xtr[ix], Ytr[ix]  # (batch_size, blockSize), (batch_size, blockSize)

    # forward pass
    logits = model(Xb)               # (batch_size, blockSize, vocab_size)
    loss = cross_entropy(logits, Yb) # loss function

    # backward pass
    # for p in parameters:
    #     p.grad = np.zeros_like(p.data, dtype=float)
    opt.zero_grad()
    loss.backward()

    lr = 1e-3 if i < 0.8 * max_steps else 1e-4
    opt.step(lr)
    # for p in parameters:
    #     p.data += -lr * p.grad

    # track stats
    lossi.append(float(loss.data))
    ud.append(opt.ud)

    if i % 30 == 0: # cheap running average of the last 200 batches, no extra compute
        print(f'{i:7d}/{max_steps:7d}: {np.mean(lossi[-200:]):.4f}')
    if i % eval_every == 0 or i == max_steps - 1:
        tr, va = split_loss(Xtr, Ytr), split_loss(Xva, Yva)
        val_lossi.append((i, tr, va))
        print(f'{i:7d}/{max_steps:7d}: train {tr:.4f}  val {va:.4f}  (gap {va-tr:+.4f})')

train windows: 3,309,373   val windows: 367,652
      0/  30000: 3.6374
      0/  30000: train 3.5781  val 3.5740  (gap -0.0041)
     30/  30000: 3.5601
     60/  30000: 3.5353
     90/  30000: 3.5290
    120/  30000: 3.5251
    150/  30000: 3.5267
    180/  30000: 3.5251
    210/  30000: 3.5254
    240/  30000: 3.5241
    270/  30000: 3.5266
    300/  30000: 3.5303
    330/  30000: 3.5304
    360/  30000: 3.5301
    390/  30000: 3.5360
    420/  30000: 3.5366
    450/  30000: 3.5382
    480/  30000: 3.5348
    510/  30000: 3.5336
    540/  30000: 3.5324
    570/  30000: 3.5281
    600/  30000: 3.5233
    630/  30000: 3.5247
    660/  30000: 3.5257
    690/  30000: 3.5254
    720/  30000: 3.5259
    750/  30000: 3.5336
    780/  30000: 3.5330
    810/  30000: 3.5341
    840/  30000: 3.5289
    870/  30000: 3.5275
    900/  30000: 3.5259
    930/  30000: 3.5263
    960/  30000: 3.5225
    990/  30000: 3.5296
   1000/  30000: train 3.5137  val 3.5773  (gap +0.0635)
   1020/  30000: 3.528

KeyboardInterrupt: 

In [ ]:
# sample from the model
rng = np.random

out = []
context = [0] * blockSize
while len(out) < 10000:
    x = np.array([context]) # (1, blockSize); the Embedding layer looks these up
    logits = model(x)                     # (1, blockSize, vocab_size): a prediction at every position
    last_logits = logits.data[:, -1, :]    # only the newest position matters for generation
    e = np.exp(last_logits - last_logits.max(axis=1, keepdims=True))
    probs = e / e.sum(axis=1, keepdims=True)
    # sample from the distribution
    ix = rng.choice(vocab_size, p=probs[0])
    if ix == EOS_ID:
        break  # the model signaled the end of its own generation
    # shift the context window and track the samples
    context = context[1:] + [ix]
    out.append(ix)


print(tok.decode(out))

In [7]:
# encode a query and use it as the seed context, instead of starting from all padding
rng = np.random

query = input("Query: ")
query_tokens = encode(query)

# left-pad with the pad token (0) if the prompt is shorter than blockSize,
# or keep only the most recent blockSize tokens if it's longer
if len(query_tokens) < blockSize:
    context = [0] * (blockSize - len(query_tokens)) + query_tokens
else:
    context = query_tokens[-blockSize:]

print("Thinking...")

num_new_tokens = 300  # hard cap in case the model never emits EOS_ID
out = []  # only the newly generated tokens - the query itself gets cut from the printed response

for _ in range(num_new_tokens):
    x = np.array([context])              # (1, blockSize); the Embedding layer looks these up
    logits = model(x)                    # (1, blockSize, vocab_size): a prediction at every position
    last_logits = logits.data[:, -1, :]  # only the newest position matters for generation
    e = np.exp(last_logits - last_logits.max(axis=1, keepdims=True))
    probs = e / e.sum(axis=1, keepdims=True)
    # sample from the distribution
    ix = rng.choice(vocab_size, p=probs[0])
    if ix == EOS_ID:
        break  # the model signaled it's done responding
    # shift the context window and track the samples
    context = context[1:] + [ix]
    out.append(ix)

print(f"\nQuery: {query}\nResponse: {tok.decode(out)}")

Thinking...

Query: What's 2 * 4?
Response:  Due to accurawards, level his can a fairly enterm gibris conney experiencing by Kims OClore in containly original exists and ram semine and shareding study groups for these confecting the directors.
Pritiannnnection a recent to make the loadies with fruits from sessionards were being traidness to tell her as a damage olderous new export, need inherating or find only might have continued with it contest out how meaning the viewing type of the others for high confens this other pense temple. Where of some communities they also conceed with Musbileer, says, individed nine when herancel[*Y. Numaiction,
Whier Miniows, aounay of Calistrieliting of Squiv. You be used in sustains plinning. But further caper and next, Sa


In [22]:
# Basic attention bit

B, T, C = 4, 8, 2 
x = np.random.randn(B, T, C)
print(x.shape)

(4, 8, 2)


In [23]:
xbow = np.zeros((B,T,C)) # Bag of Words
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b, t] = np.mean(xprev, 0)

xbow[2]

array([[-0.83360006,  0.45040839],
       [-1.10059142, -0.4697657 ],
       [-1.03429883, -1.12052612],
       [-1.01487549, -0.6256353 ],
       [-0.60080551, -0.47944088],
       [-0.61257341, -0.46789691],
       [-0.7301151 , -0.64742479],
       [-0.64868054, -0.55582508]])

In [24]:
wei = np.tril(np.ones((T, T)))
wei = wei / wei.sum(1, keepdims=True)
xbow2 = wei @ x
xbow2[2]

array([[-0.83360006,  0.45040839],
       [-1.10059142, -0.4697657 ],
       [-1.03429883, -1.12052612],
       [-1.01487549, -0.6256353 ],
       [-0.60080551, -0.47944088],
       [-0.61257341, -0.46789691],
       [-0.7301151 , -0.64742479],
       [-0.64868054, -0.55582508]])

In [7]:
# Self attention
B,T,C = 4,8,32 # Batch, Time, Channels
x = Value(np.random.randn(B, T, C))

head = Head(C, head_size=16, block_size=T)
out = head(x)
out.shape

(4, 8, 16)

In [10]:
# Writes docs/finewebWeights.json. The shakespear model keeps docs/gptWeights.json.
#
# Do NOT call export_gpt_weights from this notebook. Its default path is the shakespear
# file, so running it against this notebook's model overwrites v3 with the byte-pair
# weights - which is exactly what happened once already. exportGPTWeights.py now refuses
# a vocab mismatch, so that mistake fails loudly instead of silently, but the right
# exporter to call here is this one.
from exportFineWebWeights import export_fineweb_weights
export_fineweb_weights(model, tok)

Wrote docs/finewebWeights.json (1270.6 KB) - 119,169 params, vocab 1025, n_embd 32, 4 blocks x 4 heads, block size 64, 768 merges, eos 1024


'docs/finewebWeights.json'